<a href="https://colab.research.google.com/github/yaeldorin-glitch/Applied-Machine-Learning/blob/main/colorization_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================
# Part II - Image Colorization - TEMPLATE
# =====================================================================
#
# Task: take a grayscale image and predict its colors.
#
# You build and train the model any way you want (autoencoder, VAE, GAN, ...).
#
# The grader will:
#   1. run your model class + load_model() to load your saved weights,
#   2. call colorize() on their own images,
#   3. compare your output to hidden color images (PSNR / MSE).
#
# So you MUST keep the 3 fixed rules below.
# =====================================================================
#
# ---------------------------------------------------------------------
# FIXED RULES (do not change)
# ---------------------------------------------------------------------
#
# 1. Save your trained weights as a state_dict:
#        torch.save(model.state_dict(), "weights.pth")
#    Submit this "weights.pth" file together with your notebook.
#
# 2. All images are 256 x 256 PNG.
#    colorize() input  : grayscale array, shape (256, 256), float in [0, 1]
#    colorize() output : RGB array,       shape (256, 256, 3), float in [0, 1]
#
# 3. Do file loading OUTSIDE colorize() (see the demo at the bottom).
#    colorize() only takes arrays, not file paths.
# ---------------------------------------------------------------------

In [ ]:
import numpy as np
import torch
import torch.nn as nn

IMG_SIZE = 256

In [ ]:
# ---------------------------------------------------------------------
# 1) YOUR MODEL
# ---------------------------------------------------------------------
# A U-Net that predicts only CHROMINANCE (Cb, Cr), not full RGB.
#
# Plain RGB regression with L1/MSE loss is known to produce muted,
# undersaturated colors: the model hedges toward "safe" averaged
# colors whenever it's unsure, because errors in R/G/B are entangled
# with brightness errors too. The standard fix in the colorization
# literature is to work in a luma/chroma colorspace (Y/Cb/Cr): the
# grayscale input IS (almost exactly) the Y channel already, so it
# doesn't need to be predicted at all -- only the 2 color channels
# (Cb, Cr) do. This confines all prediction error to color, never
# brightness, which is both an easier learning problem and closer to
# how the eye actually perceives color images.
#
# base=24 (~4.4M params): with Part I finished, the full machine is
# available, so this affords a bigger network than the base=16 first
# attempt. The default here MUST match what's actually trained --
# load_model() below rebuilds with no arguments, so a mismatched
# default would fail to load the saved weights.

Y_R, Y_G, Y_B = 0.299, 0.587, 0.114  # matches PIL's L = ITU-R 601-2 luma


def rgb_to_ycbcr(rgb):
    """rgb: (..., H, W, 3) in [0,1] -> y, cb, cr each (..., H, W) in [0,1]."""
    r, g, b = rgb[..., 0], rgb[..., 1], rgb[..., 2]
    y = Y_R * r + Y_G * g + Y_B * b
    cb = -0.168736 * r - 0.331264 * g + 0.5 * b + 0.5
    cr = 0.5 * r - 0.418688 * g - 0.081312 * b + 0.5
    return y, cb, cr


def ycbcr_to_rgb(y, cb, cr):
    """y, cb, cr: (..., H, W) in [0,1] -> rgb (..., H, W, 3) in [0,1]."""
    cb0, cr0 = cb - 0.5, cr - 0.5
    r = y + 1.402 * cr0
    g = y - 0.344136 * cb0 - 0.714136 * cr0
    b = y + 1.772 * cb0
    return np.clip(np.stack([r, g, b], axis=-1), 0.0, 1.0)


def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
    )


class ColorizeModel(nn.Module):
    def __init__(self, base=24):
        super().__init__()
        self.enc1 = conv_block(1, base)
        self.enc2 = conv_block(base, base * 2)
        self.enc3 = conv_block(base * 2, base * 4)
        self.enc4 = conv_block(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = conv_block(base * 8, base * 16)

        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.dec4 = conv_block(base * 16, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = conv_block(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = conv_block(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = conv_block(base * 2, base)

        self.out = nn.Conv2d(base, 2, 1)  # Cb, Cr only

    def forward(self, x):
        # x: (batch, 1, 256, 256) -- the Y (luminance) channel
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return torch.sigmoid(self.out(d1))  # (batch, 2, 256, 256): cb, cr in [0,1]

In [ ]:
# ---------------------------------------------------------------------
# 2) LOAD YOUR TRAINED WEIGHTS
# ---------------------------------------------------------------------
# Rebuilds the empty model and loads the saved numbers.
# This is instant - no training.
def load_model(weights_path="weights.pth"):
    model = ColorizeModel()
    model.load_state_dict(torch.load(weights_path, map_location="cpu"))
    model.eval()
    return model

In [ ]:
# ---------------------------------------------------------------------
# 3) COLORIZE ONE IMAGE
# ---------------------------------------------------------------------
# gray_img: numpy array, shape (256, 256), float in [0, 1]
# returns : numpy array, shape (256, 256, 3), float in [0, 1]
#
# The external contract is unchanged (grayscale in, RGB out) -- the
# Y/Cb/Cr split is purely an internal implementation detail. gray_img
# is used directly as Y (it already IS the luminance channel), the
# model predicts Cb/Cr, and the three are recombined into RGB.
def colorize(gray_img, model):
    x = torch.from_numpy(gray_img).float().view(1, 1, IMG_SIZE, IMG_SIZE)

    with torch.no_grad():           # no gradients needed for inference
        cb_cr = model(x)            # (1, 2, 256, 256)

    cb = cb_cr[0, 0].cpu().numpy()
    cr = cb_cr[0, 1].cpu().numpy()
    return ycbcr_to_rgb(gray_img, cb, cr)

In [ ]:
# =======================================================================
# TRAINING -- produces weights.pth
# =======================================================================
# Everything above this line is the fixed submission interface. Below
# is how weights.pth was actually produced: dataset, training loop,
# and a quick self-check against the same PSNR/MSE metric the grader
# uses, before the final demo.
#
# No training images were provided for this assignment ("you may
# choose all the images you want to train your model" -- directives.txt).
# Flowers102 (torchvision, auto-downloads) was used: colorful, diverse
# natural photos, no license friction, no manual collection needed.
# Only the images are used -- the 102 flower classes are irrelevant to
# colorization and are never touched.

import os
import glob
import random
import torchvision
import torchvision.transforms as T
from PIL import Image

DATA_ROOT = "colorization_data"
os.makedirs(DATA_ROOT, exist_ok=True)
for split in ["train", "val", "test"]:
    torchvision.datasets.Flowers102(root=DATA_ROOT, split=split, download=True)
IMAGE_DIR = os.path.join(DATA_ROOT, "flowers-102", "jpg")

image_paths = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.jpg")))
print("images found:", len(image_paths))

random.Random(0).shuffle(image_paths)
# 1500 of the 8189 available images: enough diversity while still
# letting the model see each image often enough (15 epochs) to
# converge -- for a fairly narrow, repetitive domain like flower
# photos, more images x fewer epochs was measured to train worse than
# fewer images x more epochs at a fixed compute budget.
N_IMAGES = 1500
image_paths = image_paths[:N_IMAGES]
n_val = max(1, int(0.1 * len(image_paths)))
val_paths, train_paths = image_paths[:n_val], image_paths[n_val:]
print("train:", len(train_paths), " val:", len(val_paths))


class ColorizationDataset(torch.utils.data.Dataset):
    """Loads a color photo, returns (Y, [Cb, Cr]), all 256x256 in [0, 1].

    Y is the grayscale input (== luminance); Cb/Cr are the color
    channels the model must predict. The photo supervises itself --
    no separate labels needed. `augment=True` applies a random
    horizontal flip (train split only) for free extra variety from
    the same images.
    """

    def __init__(self, paths, size=IMG_SIZE, augment=False):
        self.paths = paths
        self.resize = T.Resize((size, size))
        self.augment = augment

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = self.resize(Image.open(self.paths[idx]).convert("RGB"))
        if self.augment and random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        rgb = np.array(img).astype("float32") / 255.0
        y, cb, cr = rgb_to_ycbcr(rgb)
        y_t = torch.from_numpy(y).float().unsqueeze(0)
        cbcr_t = torch.from_numpy(np.stack([cb, cr], axis=0)).float()
        return y_t, cbcr_t


train_ds = ColorizationDataset(train_paths, augment=True)
val_ds = ColorizationDataset(val_paths, augment=False)

images found: 8189
train: 1350  val: 150


In [ ]:
BATCH_SIZE = 8
EPOCHS = 15
LR = 1e-3

torch.set_num_threads(4)  # balanced alongside Part I's own 4 threads
torch.manual_seed(0)
model = ColorizeModel()
opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.L1Loss()  # L1 over MSE: sharper edges, less blurring

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Using device: {device}")

import time
print("Starting training timer...")
start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for y, cbcr in train_loader:
        y, cbcr = y.to(device), cbcr.to(device) # Move data to device
        opt.zero_grad()
        pred = model(y)
        loss = loss_fn(pred, cbcr)
        loss.backward()
        opt.step()
        train_loss += loss.item() * y.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for y, cbcr in val_loader:
            y, cbcr = y.to(device), cbcr.to(device) # Move data to device
            val_loss += loss_fn(model(y), cbcr).item() * y.size(0)
    val_loss /= len(val_ds)

    print("epoch %2d | train L1 %.4f | val L1 %.4f" % (epoch, train_loss, val_loss))

end_time = time.time()
training_duration = end_time - start_time
print(f"Total training time: {training_duration:.2f} seconds")

print("Saving weights...")
torch.save(model.state_dict(), "weights.pth")
print("weights saved to weights.pth")

Using device: cuda
Starting training timer...
epoch  0 | train L1 0.0632 | val L1 0.0604


In [ ]:
# Self-check against the same metric the grader uses (PSNR), via the
# actual submission interface (load_model + colorize) -- not a
# shortcut through the training-time model object.
def psnr(pred, target, max_val=1.0):
    mse = float(np.mean((pred - target) ** 2))
    if mse == 0:
        return float("inf")
    return 10.0 * np.log10((max_val ** 2) / mse)


loaded = load_model("weights.pth")
psnrs = []
for y, cbcr in val_ds:
    y_np = y.squeeze(0).numpy()
    ground_truth_rgb = ycbcr_to_rgb(y_np, cbcr[0].numpy(), cbcr[1].numpy())
    pred_rgb = colorize(y_np, loaded)
    psnrs.append(psnr(pred_rgb, ground_truth_rgb))
print("mean val PSNR: %.2f dB over %d held-out images" % (float(np.mean(psnrs)), len(psnrs)))

In [ ]:
# =======================================================================
# DEMO -- file loading happens here, OUTSIDE colorize() (fixed rule 3)
# =======================================================================
import matplotlib.pyplot as plt

model = load_model("weights.pth")

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for row, idx in enumerate([0, 1, 2]):
    y_sample, cbcr_sample = val_ds[idx]
    y_np = y_sample.squeeze(0).numpy()
    ground_truth_rgb = ycbcr_to_rgb(y_np, cbcr_sample[0].numpy(), cbcr_sample[1].numpy())
    pred = colorize(y_np, model)

    axes[row, 0].imshow(y_np, cmap="gray")
    axes[row, 1].imshow(pred)
    axes[row, 2].imshow(ground_truth_rgb)
    if row == 0:
        for ax, title in zip(axes[row], ["input (gray)", "predicted color", "ground truth"]):
            ax.set_title(title)
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout()
plt.show()